In [ ]:
import queue
import time

import numpy as np
import tritonclient.grpc as grpcclient
import matplotlib.pyplot as plt
from IPython.display import Audio, display

TRITON_URL = "localhost:8001"
MODEL_NAME = "easymp"
SAMPLE_RATE = 22050  # codec output_sample_rate

In [ ]:
def synthesize(text: str) -> np.ndarray:
    """Send a TTS request and collect streamed audio chunks."""
    result_q: queue.Queue = queue.Queue()

    def _on_response(result, error):
        result_q.put((result, error))

    client = grpcclient.InferenceServerClient(url=TRITON_URL)
    client.start_stream(callback=_on_response)

    text_input = grpcclient.InferInput("text", [1, 1], "BYTES")
    text_input.set_data_from_numpy(np.array([[text]], dtype=object))

    client.async_stream_infer(
        model_name=MODEL_NAME,
        inputs=[text_input],
        outputs=[grpcclient.InferRequestedOutput("audio")],
    )

    chunks = []
    t0 = time.perf_counter()
    t_first = None

    while True:
        result, error = result_q.get(timeout=120)
        if error:
            client.stop_stream()
            raise RuntimeError(str(error))

        audio = result.as_numpy("audio").squeeze()
        if audio.size > 0:
            if t_first is None:
                t_first = time.perf_counter()
            chunks.append(audio)

        response = result.get_response()
        final_param = response.parameters.get("triton_final_response")
        if final_param and getattr(final_param, "bool_param", False):
            break

    client.stop_stream()

    elapsed = time.perf_counter() - t0
    ttfa = (t_first - t0) if t_first else elapsed
    print(f"Received {len(chunks)} chunks | TTFA: {ttfa:.3f}s | total: {elapsed:.3f}s")
    return np.concatenate(chunks) if chunks else np.array([], dtype=np.float32)

In [ ]:
audio = synthesize(
    text="Since then physicists have found that it is not reflection, but refraction by the raindrops which causes the rainbows.",
)
print(f"Got {len(audio)} samples — {len(audio) / SAMPLE_RATE:.2f}s @ {SAMPLE_RATE} Hz")

In [ ]:
t = np.arange(len(audio)) / SAMPLE_RATE

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, audio, linewidth=0.3)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("EasyMP Waveform")
fig.tight_layout()
plt.show()

display(Audio(audio, rate=SAMPLE_RATE))